# YOLO: валидация и разбор ошибок

После обучения важно не только посмотреть итоговую метрику, но и понять, где модель ошибается: по каким классам, с какими confidence и на каких типах изображений. Этот ноутбук — стартовая точка для такого разбора.


In [ ]:

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO


In [ ]:

# Укажите путь к весам после обучения.
# Чаще всего best.pt лежит в runs/detect/.../weights/best.pt или аналогичной директории.
MODEL_WEIGHTS = "runs_train/exp_yolo/weights/best.pt"
DATASET_YAML = "dataset/dataset.yaml"


## Загрузка модели


In [ ]:

weights_path = Path(MODEL_WEIGHTS)

if weights_path.exists():
    model = YOLO(str(weights_path))
    print("Модель загружена")
else:
    print("Файл весов не найден. Укажите корректный путь к best.pt")


## Валидация

Ниже пример запуска встроенной валидации YOLO. Она считает основные метрики и сохраняет артефакты в `runs/`.


In [ ]:

# Эта ячейка выполняет настоящую валидацию.
# Запускайте её после обучения.

# if 'model' in locals():
#     metrics = model.val(data=DATASET_YAML, split='val')
#     print(metrics)


## Идеи для разбора ошибок

На практике полезно разбирать минимум 4 группы проблем:

1. **False positives** — модель видит объект там, где его нет.
2. **False negatives** — модель пропускает объект.
3. **Путаница классов** — один класс принимается за другой.
4. **Низкий confidence** — модель не уверена даже при правильной локализации.


In [ ]:

# Ниже пример ручного инференса на валидационной папке.
# Он полезен, когда нужно быстро просмотреть картинки глазами.

VAL_IMAGES_DIR = Path("dataset/images/val")
REVIEW_OUTPUT_DIR = "runs_review"

# if 'model' in locals() and VAL_IMAGES_DIR.exists():
#     review_results = model.predict(
#         source=str(VAL_IMAGES_DIR),
#         conf=0.25,
#         save=True,
#         project=REVIEW_OUTPUT_DIR,
#         name="val_predictions",
#         exist_ok=True,
#     )
#     print("Изображений обработано:", len(review_results))


## Простая таблица для ручного контроля

Иногда удобно завести таблицу, куда заносить проблемные файлы: какие классы перепутаны, что именно не так, какие идеи по улучшению датасета.


In [ ]:

error_log_columns = [
    "image_name",
    "issue_type",      # false_positive / false_negative / class_confusion / low_confidence
    "predicted_class",
    "expected_class",
    "comment",
]

error_log = pd.DataFrame(columns=error_log_columns)
error_log


## Что обычно помогает улучшить качество

- добавить больше разнообразных примеров
- выровнять дисбаланс классов
- улучшить качество разметки
- повысить качество негативных примеров
- попробовать другой размер модели
- увеличить разрешение `imgsz`
- подобрать `conf` для продакшн-инференса отдельно от обучения


## Минимальный цикл работы

1. Прогнать baseline на `yolov8n.pt`
2. Проверить визуально 20–50 примеров
3. Исправить ошибки разметки
4. Дообучить модель
5. Снова посмотреть на проблемные классы

Этого уже достаточно для быстрой и практичной итерации.
